# Custo-Benefício de Jogos na Steam: Uma Análise Exploratória do Valor Percebido pelo Jogador.

### OBJETIVO
Analisar os jogos pagos disponíveis na Steam com o propósito de comparar o valor percebido dos títulos em relação ao custo por hora jogada, à avaliação dos usuários e à popularidade estimada, do ponto de vista do jogador interessado em tomar decisões de compra mais informadas, no contexto de uma análise exploratória baseada em dados públicos da plataforma Steam

## Importar libs

In [46]:
%pip install pandas numpy matplotlib

Note: you may need to restart the kernel to use updated packages.


In [47]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.


In [48]:
%pip install fastparquet

Note: you may need to restart the kernel to use updated packages.


In [49]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

import warnings
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


pandas: 3.0.2
numpy: 2.4.4


## Carregar Dataset Bronze

O dataset contém informações sobre jogos da plataforma Steam, incluindo variáveis como preço, tempo médio de jogo, avaliações dos usuários e estimativa de popularidade.

O objetivo desta etapa é:
- definir o caminho do arquivo
- carregar os dados em um DataFrame
- visualizar as primeiras linhas para uma inspeção inicial

In [50]:
CAMINHO_CSV = "dataset_bronze.csv"
df = pd.read_csv(CAMINHO_CSV)
df.head()

,appid,name,developer,publisher,score_rank,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu
0,730,Counter-Strike: Global Offensive,Valve,Valve,NaN,7642084,1173003,0,"100,000,000 .. 200,000,000",33852,708,6645,301,0,0,0,1013936
1,1172470,Apex Legends,Respawn,Electronic Arts,NaN,668053,326926,0,"100,000,000 .. 200,000,000",10506,496,935,246,0,0,0,124262
2,578080,PUBG: BATTLEGROUNDS,PUBG Corporation,"KRAFTON, Inc.",NaN,1520457,1037487,0,"100,000,000 .. 200,000,000",23165,717,5622,261,0,0,0,314682
3,1623730,Palworld,Pocketpair,Pocketpair,NaN,358266,22443,0,"50,000,000 .. 100,000,000",3854,835,2213,257,2999,2999,0,18028
4,440,Team Fortress 2,Valve,Valve,NaN,1044264,117208,0,"50,000,000 .. 100,000,000",21244,736,4262,102,0,0,0,43819


## Estrutura Inicial do DATASET

Inspeção estrutural do dataset bronze, analisando dimensões, nomes das colunas, tipos de dados e possíveis valores ausentes. Essa verificação permitiu compreender a organização do conjunto de dados e identificar elementos que necessitam de tratamento nas etapas seguintes do processo de preparação.

In [51]:
df.shape

(10000, 17)

In [52]:
df.columns

Index(['appid', 'name', 'developer', 'publisher', 'score_rank', 'positive',
       'negative', 'userscore', 'owners', 'average_forever', 'average_2weeks',
       'median_forever', 'median_2weeks', 'price', 'initialprice', 'discount',
       'ccu'],
      dtype='str')

In [53]:
df.dtypes

appid                int64
name                   str
developer              str
publisher              str
score_rank         float64
positive             int64
negative             int64
userscore            int64
owners                 str
average_forever      int64
average_2weeks       int64
median_forever       int64
median_2weeks        int64
price                int64
initialprice         int64
discount             int64
ccu                  int64
dtype: object

In [54]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   appid            10000 non-null  int64  
 1   name             10000 non-null  str    
 2   developer        9967 non-null   str    
 3   publisher        9946 non-null   str    
 4   score_rank       5 non-null      float64
 5   positive         10000 non-null  int64  
 6   negative         10000 non-null  int64  
 7   userscore        10000 non-null  int64  
 8   owners           10000 non-null  str    
 9   average_forever  10000 non-null  int64  
 10  average_2weeks   10000 non-null  int64  
 11  median_forever   10000 non-null  int64  
 12  median_2weeks    10000 non-null  int64  
 13  price            10000 non-null  int64  
 14  initialprice     10000 non-null  int64  
 15  discount         10000 non-null  int64  
 16  ccu              10000 non-null  int64  
dtypes: float64(1), int64(12)

## Tratamento de Dados Ausentes

Foi realizada a identificação de valores ausentes presentes no conjunto de dados, com o objetivo de avaliar a completude das variáveis e identificar possíveis inconsistências que possam impactar as análises posteriores. Para isso, foram verificadas a quantidade e a proporção de registros nulos em cada coluna do dataset.

In [55]:
ausentes = pd.DataFrame({
    'qtd_ausentes': df.isna().sum(),
    'percentual_ausentes': df.isna().mean() * 100
})

ausentes.sort_values(by='qtd_ausentes', ascending=False)

,qtd_ausentes,percentual_ausentes
score_rank,9995,99.95
publisher,54,0.54
developer,33,0.33
appid,0,0.00
name,0,0.00
positive,0,0.00
negative,0,0.00
userscore,0,0.00
owners,0,0.00
average_forever,0,0.00


A coluna ```score_rank``` foi removida durante a etapa de limpeza por apresentar quantidade elevada de valores ausentes, com apenas 5 valores preenchidos em 10.000 registros. Isso representa 9.995 registros nulos e menos de 0,1% de completude. Além disso, esse campo não contribui diretamente para responder às questões investigáveis do projeto.

```
A coluna score_rank possui:
- apenas 5 valores preenchidos;
- 9995 valores ausentes;
- menos de 0,1% de completude
```

In [56]:
total_registros = len(df)
nulos_score_rank = df['score_rank'].isna().sum()
preenchidos_score_rank = df['score_rank'].notna().sum()
percentual_nulos_score_rank = df['score_rank'].isna().mean() * 100
percentual_preenchidos_score_rank = df['score_rank'].notna().mean() * 100

print(f'Total de registros: {total_registros}')
print(f'Valores preenchidos em score_rank: {preenchidos_score_rank}')
print(f'Valores ausentes em score_rank: {nulos_score_rank}')
print(f'Percentual de valores ausentes: {percentual_nulos_score_rank:.2f}%')
print(f'Percentual de completude: {percentual_preenchidos_score_rank:.2f}%')

Total de registros: 10000
Valores preenchidos em score_rank: 5
Valores ausentes em score_rank: 9995
Percentual de valores ausentes: 99.95%
Percentual de completude: 0.05%


In [57]:
df = df.drop(columns=['score_rank'])

Apesar da presença de valores ausentes nas colunas `developer` e `publisher`, optou-se por manter essas variáveis no conjunto de dados, uma vez que a quantidade de registros nulos é relativamente baixa em relação ao total de observações e não compromete diretamente as métricas e questões analíticas definidas no projeto.

## Verificação de Registros Duplicados

Realizada a verificação de registros duplicados no conjunto de dados, com o objetivo de identificar possíveis redundâncias ou inconsistências que possam comprometer a qualidade das análises. A análise considerou tanto duplicidades completas entre linhas quanto possíveis repetições do identificador único dos jogos (`appid`), que deveria apresentar valores exclusivos para cada registro do dataset.

In [58]:
df.duplicated().sum() #Verificação das linhas duplicadas

np.int64(50)

In [59]:
df['appid'].duplicated().sum()

np.int64(50)

Verificação das linhas duplicadas para verificar se os registros realmente são idênticos, sem diferenças relevantes.

In [60]:
df[df['appid'].duplicated(keep=False)].sort_values(by='appid')

,appid,name,developer,publisher,positive,negative,userscore,owners,average_forever,average_2weeks,median_forever,median_2weeks,price,initialprice,discount,ccu
9007,3380,Dynomite Deluxe,"PopCap Games, Inc.","PopCap Games, Inc.",334,17,0,"100,000 .. 200,000",76,0,112,0,499,499,0,0
8989,3380,Dynomite Deluxe,"PopCap Games, Inc.","PopCap Games, Inc.",334,17,0,"100,000 .. 200,000",76,0,112,0,499,499,0,0
7005,42980,Victoria I Complete,Paradox Development Studio,Paradox Interactive,180,72,0,"100,000 .. 200,000",159,0,94,0,999,999,0,3
6992,42980,Victoria I Complete,Paradox Development Studio,Paradox Interactive,180,72,0,"100,000 .. 200,000",159,0,94,0,999,999,0,3
6001,70100,Hacker Evolution,exosyphen studios,exosyphen studios,710,337,0,"100,000 .. 200,000",210,0,207,0,299,299,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6011,2458530,魔女的夜宴,YUZUSOFT,HIKARI FIELD,8658,117,0,"100,000 .. 200,000",742,44,551,87,3499,3499,0,295
7986,2694420,KILL KNIGHT,PlaySide,PlaySide,1485,91,0,"100,000 .. 200,000",152,0,84,0,1499,1499,0,12
8008,2694420,KILL KNIGHT,PlaySide,PlaySide,1485,91,0,"100,000 .. 200,000",152,0,84,0,1499,1499,0,12
8990,3535050,Void Breaker,AA Mini Games Studio,AA Mini Games Studio,2,0,0,"100,000 .. 200,000",0,0,0,0,99,99,0,0


A verificação de duplicidades identificou a existência de 50 registros repetidos no conjunto de dados, incluindo repetições do identificador único `appid`. Após inspeção dos registros duplicados, observou-se que as linhas apresentavam informações idênticas, caracterizando redundância de dados.

In [61]:
df = df.drop_duplicates()

In [62]:
df.duplicated().sum()

np.int64(0)

In [63]:
df['appid'].duplicated().sum()

np.int64(0)

Os registros duplicados foram removidos utilizando o método `drop_duplicates()`, reduzindo redundâncias e garantindo maior consistência para as etapas posteriores de análise.

## Verificação de Valores Zerados e Inconsistências

Verificação de valores zerados nas variáveis numéricas do conjunto de dados, com o objetivo de identificar possíveis inconsistências ou registros que possam impactar cálculos e métricas. Essa análise é importante para atributos relacionados a preço, avaliações e tempo de jogo, uma vez que valores iguais a zero podem influenciar diretamente indicadores como custo por hora jogada, retenção e score de custo-benefício.

In [64]:
colunas_numericas = [
    'positive',
    'negative',
    'userscore',
    'average_forever',
    'average_2weeks',
    'median_forever',
    'median_2weeks',
    'price',
    'initialprice',
    'discount',
    'ccu'
]

(df[colunas_numericas] == 0).sum()

positive             37
negative            174
userscore          9945
average_forever    1190
average_2weeks     7239
median_forever     1190
median_2weeks      7239
price              2061
initialprice       2061
discount           8653
ccu                2601
dtype: int64

## Remoção de Jogos Gratuitos

Foi realizada a remoção dos jogos com preço igual a zero, correspondentes a títulos gratuitos ou dados incompletos. Essa decisão foi adotada porque o problema analítico do projeto busca analisar custo-benefício com base na relação entre preço pago, tempo de jogo e avaliação dos usuários. Dessa forma, jogos gratuitos não se adequam às métricas propostas, especialmente ao cálculo de custo por hora jogada.

In [65]:
df = df[df['price'] > 0]

In [66]:
(df['price'] == 0).sum()

np.int64(0)

In [67]:
df.shape

(7889, 16)

## Remoção de Registros sem Tempo de Jogo

Como as métricas analíticas do projeto utilizam o tempo médio total de jogo (`average_forever`) para o cálculo de custo por hora jogada e retenção, foram removidos os registros com valor igual a zero nessa variável. Essa decisão foi adotada para evitar inconsistências matemáticas, como divisões por zero, além de excluir jogos sem informação efetiva de uso pelos jogadores.

Os registros com `average_2weeks` igual a zero foram mantidos, pois a ausência de atividade recente não caracteriza necessariamente inconsistência nos dados, podendo representar jogos antigos ou atualmente inativos.

In [68]:
df = df[df['average_forever'] > 0]

In [69]:
(df['average_forever'] == 0).sum()

np.int64(0)

## Remoção de Registros sem Avaliações

As colunas `positive` e `negative` são utilizadas no cálculo do indicador `steam_rating`, responsável por representar a proporção de avaliações positivas dos jogos. Dessa forma, foram removidos os registros em que a soma entre avaliações positivas e negativas era igual a zero, uma vez que esses casos impossibilitam o cálculo da métrica e indicam ausência total de avaliações por parte dos usuários.

In [70]:
df = df[(df['positive'] + df['negative']) > 0]

In [71]:
((df['positive'] + df['negative']) == 0).sum()

np.int64(0)

In [72]:
df.shape

(7166, 16)

As demais variáveis numéricas do conjunto de dados foram mantidas, mesmo apresentando valores zerados em parte dos registros. Atributos como `discount` e `ccu`, por exemplo, não serão utilizados diretamente nas métricas e análises centrais do projeto. Dessa forma, a remoção de registros com valores iguais a zero nessas colunas poderia eliminar linhas potencialmente válidas para as variáveis efetivamente utilizadas nas métricas analíticas, como preço, tempo de jogo e avaliações dos usuários. Por esse motivo, optou-se por preservar esses registros nesta etapa do processo de limpeza.

## Verificação da Variável Owners

A variável `owners` será utilizada nas métricas relacionadas à popularidade dos jogos e na identificação de possíveis hidden gems. Dessa forma, foi realizada a verificação de registros sem informação válida de proprietários, uma vez que esses casos podem comprometer cálculos posteriores envolvendo percentis e classificação analítica dos jogos.

In [73]:
df['owners'].unique()

<ArrowStringArray>
['50,000,000 .. 100,000,000',  '20,000,000 .. 50,000,000',
  '10,000,000 .. 20,000,000',   '5,000,000 .. 10,000,000',
    '2,000,000 .. 5,000,000',    '1,000,000 .. 2,000,000',
      '500,000 .. 1,000,000',        '200,000 .. 500,000',
        '100,000 .. 200,000',         '50,000 .. 100,000']
Length: 10, dtype: str

In [74]:
df['owners'].value_counts().head(20)

owners
100,000 .. 200,000           2478
200,000 .. 500,000           2254
500,000 .. 1,000,000          939
1,000,000 .. 2,000,000        600
50,000 .. 100,000             406
2,000,000 .. 5,000,000        334
5,000,000 .. 10,000,000        90
10,000,000 .. 20,000,000       40
20,000,000 .. 50,000,000       20
50,000,000 .. 100,000,000       5
Name: count, dtype: int64

A análise da variável `owners` não identificou registros zerados, valores ausentes ou categorias inválidas. As informações de proprietários estão representadas por faixas numéricas textuais, indicando estimativas de popularidade dos jogos na plataforma Steam. Dessa forma, não foi necessária a remoção de registros nessa variável durante a etapa de limpeza.

## Identificação de Outliers

Realizada a verificação de possíveis outliers nas variáveis numéricas relevantes para as análises do projeto. A identificação de outliers é importante porque valores extremamente altos ou baixos podem influenciar métricas derivadas, normalizações e comparações estatísticas, afetando a interpretação dos resultados analíticos.

Para essa verificação, foi utilizado o método do Intervalo Interquartil (IQR), que identifica registros situados abaixo ou acima dos limites estatísticos definidos a partir dos quartis da distribuição dos dados.

In [75]:
def identificar_outliers_iqr(dataframe, nome_coluna):
    primeiro_quartil = dataframe[nome_coluna].quantile(0.25)
    terceiro_quartil = dataframe[nome_coluna].quantile(0.75)

    intervalo_interquartil = terceiro_quartil - primeiro_quartil

    limite_inferior = primeiro_quartil - 1.5 * intervalo_interquartil
    limite_superior = terceiro_quartil + 1.5 * intervalo_interquartil

    outliers = dataframe[
        (dataframe[nome_coluna] < limite_inferior) |
        (dataframe[nome_coluna] > limite_superior)
    ]

    return outliers

In [76]:
colunas_para_outliers = [
    'price',
    'positive',
    'negative',
    'average_forever',
    'average_2weeks'
]

for coluna in colunas_para_outliers:
    outliers = identificar_outliers_iqr(df, coluna)

    print(f'\nColuna: {coluna}')
    print(f'Quantidade de outliers: {outliers.shape[0]}')

    display(
        outliers[['name', coluna]]
        .sort_values(by=coluna, ascending=False)
        .head(20)
    )


Coluna: price
Quantidade de outliers: 305


,name,price
4836,VEGAS Pro 18 Edit Steam Edition,14999
4579,Clickteam Fusion 2.5,9999
8603,Pixel Game Maker MV,8499
2929,RPG Maker MV,7999
4728,RPG Maker MZ,7999
202,EA SPORTS FC 25,6999
8620,RPG Maker VX Ace,6999
8313,AI＊Shoujo/AI＊少女,6999
8189,EA SPORTS Madden NFL 25,6999
580,Dragon's Dogma 2,6999



Coluna: positive
Quantidade de outliers: 974


,name,positive
17,Terraria,1373979
24,Tom Clancy's Rainbow Six Siege,1172854
22,Garry's Mod,1122546
7,Black Myth: Wukong,1111720
25,Rust,1071135
16,ELDEN RING,981540
9,Left 4 Dead 2,940221
19,Wallpaper Engine,876898
28,Stardew Valley,872384
45,Euro Truck Simulator 2,844480



Coluna: negative
Quantidade de outliers: 955


,name,negative
5,Call of Duty: Modern Warfare II,294520
14,HELLDIVERS 2,241752
24,Tom Clancy's Rainbow Six Siege,225730
38,Dead by Daylight,166993
58,Battlefield 2042,158588
25,Rust,156649
37,Cyberpunk 2077,131850
27,ARK: Survival Evolved,117993
439,鬼谷八荒 Tale of Immortal,106084
95,DayZ,99903



Coluna: average_forever
Quantidade de outliers: 740


,name,average_forever
2734,懒人修仙传,263340
1456,YoloMouse - Cursor Changer,108366
4457,Drift racing car,77788
3076,XSOverlay,70270
4465,SAO Utils: Beta,62903
8421,RutonyChat,51907
1507,OVR Advanced Settings,49677
6725,Time Warpers,45486
262,Crosshair X,41126
6019,Trainz: A New Era,38614



Coluna: average_2weeks
Quantidade de outliers: 1384


,name,average_2weeks
3901,Rebel Inc: Escalation,9872
1712,CPUCores :: Maximize Your FPS,8584
2865,The Slormancer,8011
6982,Sword and Fairy 3,7890
3671,Slice & Dice,6291
8184,Wild Terra 2: New Lands,6092
4465,SAO Utils: Beta,4455
3076,XSOverlay,4175
3567,SpellForce - Platinum Edition,4061
2842,Soulmask,3822


## Remoção de Colunas Não Utilizadas

Foram removidas as colunas `discount`, `ccu`, `initialprice`, `userscore`, `median_forever` e `median_2weeks`, pois elas não serão utilizadas diretamente nas métricas e questões analíticas definidas no projeto. A remoção dessas variáveis contribui para simplificar a estrutura do dataset de trabalho, mantendo apenas os atributos necessários para as próximas etapas de preparação e análise.

In [77]:
df = df.drop(columns=[
    'discount',
    'ccu',
    'initialprice',
    'median_forever',
    'median_2weeks',
    'userscore'
])

In [78]:
df.columns

Index(['appid', 'name', 'developer', 'publisher', 'positive', 'negative',
       'owners', 'average_forever', 'average_2weeks', 'price'],
      dtype='str')

## Verificação Final da Limpeza

Após as etapas de limpeza, remoção de duplicidades, tratamento de registros inconsistentes e exclusão de variáveis não utilizadas nas análises, foi realizada uma verificação final da estrutura do dataset.


In [79]:
df.info()

<class 'pandas.DataFrame'>
Index: 7166 entries, 3 to 9999
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   appid            7166 non-null   int64
 1   name             7166 non-null   str  
 2   developer        7141 non-null   str  
 3   publisher        7119 non-null   str  
 4   positive         7166 non-null   int64
 5   negative         7166 non-null   int64
 6   owners           7166 non-null   str  
 7   average_forever  7166 non-null   int64
 8   average_2weeks   7166 non-null   int64
 9   price            7166 non-null   int64
dtypes: int64(6), str(4)
memory usage: 1.1 MB


In [80]:
df.duplicated().sum()

np.int64(0)

In [81]:
df.columns

Index(['appid', 'name', 'developer', 'publisher', 'positive', 'negative',
       'owners', 'average_forever', 'average_2weeks', 'price'],
      dtype='str')

In [82]:
df.shape

(7166, 10)

In [83]:
df.describe()

,appid,positive,negative,average_forever,average_2weeks,price
count,7.166000e+03,7.166000e+03,7166.000000,7166.000000,7166.000000,7166.000000
mean,8.312383e+05,1.291433e+04,1730.454507,1029.250768,106.659085,1504.619732
std,7.028078e+05,5.729950e+04,8188.610636,4235.226678,381.956677,1324.385895
min,1.000000e+01,2.000000e+00,0.000000,1.000000,0.000000,27.000000
25%,3.063800e+05,6.960000e+02,150.000000,218.000000,0.000000,499.000000
50%,5.729750e+05,2.024500e+03,354.500000,387.000000,0.000000,999.000000
75%,1.225568e+06,6.252750e+03,971.000000,907.000000,41.000000,1999.000000
max,3.431040e+06,1.373979e+06,294520.000000,263340.000000,9872.000000,14999.000000


## Geração do Dataset Silver

Após as etapas de diagnóstico, limpeza e padronização dos dados, o conjunto resultante foi salvo no formato Parquet, originando a camada silver do pipeline analítico. Essa versão representa um dataset mais consistente e adequado para as próximas etapas de transformação e construção da camada analítica (gold).

In [84]:
df.to_parquet('dataset_silver.parquet', index=False, engine='pyarrow')

In [85]:
df_silver = pd.read_parquet('dataset_silver.parquet', engine='fastparquet')

df_silver.head()

,appid,name,developer,publisher,positive,negative,owners,average_forever,average_2weeks,price
0,1623730,Palworld,Pocketpair,Pocketpair,358266,22443,"50,000,000 .. 100,000,000",3854,835,2999
1,1938090,Call of Duty: Modern Warfare II,"Treyarch, Raven Software, Beenox, High Moon St...",Activision,419594,294520,"50,000,000 .. 100,000,000",5397,639,3849
2,1063730,New World: Aeternum,Amazon Games,Amazon Games,196798,90080,"50,000,000 .. 100,000,000",10588,18,5999
3,2358720,Black Myth: Wukong,Game Science,Game Science,1111720,38378,"50,000,000 .. 100,000,000",3268,524,5999
4,550,Left 4 Dead 2,Valve,Valve,940221,23762,"50,000,000 .. 100,000,000",2479,352,999
